# 00 — WRDS Setup & Connection Test

**Thesis:** Implied Volatility Smile Spillovers (AP-33)  
**Author:** Başar Hacımustafaoğlu — 14866196  
**Purpose:** Verify that the WRDS Python connection works and that the expected databases are accessible under the UvA subscription.  

---

## What this notebook does

1. Installs and imports required packages
2. Connects to WRDS using credentials from `.env`
3. Lists all libraries (databases) available to your account
4. Checks specifically for the databases needed for this thesis
5. Saves a connection log to `logs/`

**This notebook does NOT pull any data. It only tests access.**

---

> ⚠️ **Before running:** make sure you have:
> 1. Copied `.env.example` → `.env` and filled in your WRDS username
> 2. Installed dependencies: `pip install -r requirements.txt`

## Step 1 — Imports

In [ ]:
import os
import sys
import datetime
import wrds
import pandas as pd
from dotenv import load_dotenv

# Add src/ to path so we can import wrds_utils
sys.path.append(os.path.join(os.getcwd(), '..', 'src'))
from wrds_utils import connect_wrds

print(f"Python     : {sys.version}")
print(f"wrds pkg   : {wrds.__version__}")
print(f"pandas     : {pd.__version__}")
print(f"Run time   : {datetime.datetime.now()}")

## Step 2 — Connect to WRDS

The first time you run this, WRDS will ask you to enter your password interactively and will offer to save a `pgpass` file so you don't have to re-enter it every time. Accept this — it is safe and standard.

In [ ]:
# Load .env credentials
load_dotenv()

# Connect
conn = connect_wrds()

## Step 3 — List all accessible libraries

This shows every database (called a 'library' in WRDS terminology) that your account can query.

In [ ]:
# List all libraries available to your account
libraries = conn.list_libraries()

print(f"Total libraries accessible: {len(libraries)}")
print("\nFull list:")
for lib in sorted(libraries):
    print(f"  {lib}")

## Step 4 — Check thesis-critical databases

We check for each database we need. Green = accessible. Red = not found.

In [ ]:
# Define the databases we need to check
# Format: (wrds_library_name, description, priority)
DATABASES_TO_CHECK = [
    # --- CORE (non-negotiable) ---
    ("optionm",         "OptionMetrics IvyDB US (full)",          "CORE"),
    ("optionmeurope",   "OptionMetrics IvyDB Europe (full)",       "CORE"),
    ("optionmsamp",     "OptionMetrics IvyDB US (sample)",         "CORE - sample"),
    ("optionmsampeur",  "OptionMetrics IvyDB Europe (sample)",     "CORE - sample"),
    ("cboe",            "CBOE VIX Daily",                          "CORE"),
    ("frb",             "Federal Reserve Board Rates",             "CORE"),
    ("crsp",            "CRSP Stock & Indexes",                    "CORE"),
    ("comp",            "Compustat Global Daily",                  "CORE"),
    # --- CONTROLS ---
    ("ff",              "Fama-French Factors",                     "CONTROL"),
    ("djones",          "Dow Jones Averages",                      "CONTROL"),
    ("pwt",             "Penn World Tables",                       "CONTROL"),
    ("macrofin",        "Macro Finance Society",                   "CONTROL"),
    ("markit",          "Markit CDS / CDX",                        "CONTROL"),
    ("phlx",            "PHLX Currency Options & IV",              "CONTROL"),
    # --- SUPPLEMENTARY ---
    ("wrdsapps",        "WRDS Applications (indices, linking)",    "SUPPLEMENTARY"),
    ("tr",              "Thomson Reuters / Datastream samples",    "SUPPLEMENTARY"),
]

print(f"{'Database':<20} {'Description':<45} {'Priority':<15} {'Access'}")
print("-" * 100)

results = []
for lib, desc, priority in DATABASES_TO_CHECK:
    accessible = lib in libraries
    status = "✓ ACCESSIBLE" if accessible else "✗ NOT FOUND"
    print(f"{lib:<20} {desc:<45} {priority:<15} {status}")
    results.append({"library": lib, "description": desc,
                    "priority": priority, "accessible": accessible})

results_df = pd.DataFrame(results)

## Step 5 — For accessible databases, list their tables

For each database we can access, we list the tables inside it. This tells us what data is actually available to query.

In [ ]:
# List tables for each accessible thesis-relevant database
accessible_libs = [r["library"] for r in results if r["accessible"]]

table_registry = {}

for lib in accessible_libs:
    try:
        tables = conn.list_tables(library=lib)
        table_registry[lib] = tables
        print(f"\n--- {lib} ({len(tables)} tables) ---")
        for t in sorted(tables):
            print(f"  {t}")
    except Exception as e:
        print(f"\n--- {lib} --- ERROR: {e}")
        table_registry[lib] = []

## Step 6 — Save connection log

In [ ]:
import json

timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
log_path = f"../logs/wrds_access_check_{timestamp}.json"

log = {
    "timestamp": timestamp,
    "wrds_username": os.getenv("WRDS_USERNAME"),
    "total_libraries_accessible": len(libraries),
    "thesis_database_check": results,
    "table_registry": {k: v for k, v in table_registry.items()}
}

with open(log_path, "w") as f:
    json.dump(log, f, indent=2)

print(f"Log saved to: {log_path}")

## Step 7 — Close connection

In [ ]:
conn.close()
print("WRDS connection closed.")
print("\n✓ Setup test complete. Proceed to notebook 01_data_pull_inspect.ipynb")